# Shot Map Analysis

## Barcelona vs Real Madrid

A football analytics analysis of shots locations and outcomes using StatsBomb event data.

## 1. Project Overview

This project analyzes Barcelona's shots locations and outcome in a single match against Real Madrid using publicity available StatsBomb event data.

The objective is to transforme event-level football data into a shot-level analytical dataset and use it to describe where Barcelona's attacks were finished by shots and what outcomes were obtained from those locations.

The analysis combines data validation, spatial analysis and football-specific visualization to produce a reproducible shot map.

## 2. Sporting Question

### Sporting question

> From which areas did Barcelona finish their attacks with shots, and what were the outcomes?

### Analytical objective

The objective is to describe the spatial distribution of Barcelona's shots during the match and examine how shot outcomes are distributed across the defined pitch zones.

The analysis focuses on two dimensions:

- **Shot location:** where the shot originated.
- **Shot outcome:** what happened to the shot.

### Analytical framework

| Element | Definition |
|---|---|
| Unit of analysis | Individual Barcelona shot event |
| Population | All Barcelona shot events in the analyzed match |
| Main spatial variable | Shot starting location |
| Outcome variable | Shot outcome |
| Spatial comparison | Pitch zones |

## 3. Data & Match Context

### Data source

The analysis use publicly available event-level data from StatsBomb Open Data.

StatsBomb event data records actions ocurring during a football match, including passes, carries, shots, defensive actions and other events. For this project, the analysis is restricted to shot events.

### Import, root and paremeters

In [1]:
from pathlib import Path
import json
import pandas as pd
import sys

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
EVENTS_DIR = DATA_DIR / "events"
MATCHES_DIR = DATA_DIR / "matches"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"

MATCH_ID = 69299  # Reemplazar por el ID del partido que analizaste
TEAM_NAME = "Barcelona"

### Check project paths

In [9]:
assert EVENTS_DIR.exists(), "No existe el directorio: {}".format(EVENTS_DIR)
assert MATCHES_DIR.exists(), "No existe el directorio: {}".format(MATCHES_DIR)

event_file = EVENTS_DIR / "{}.json".format(MATCH_ID)

assert event_file.exists(), "No existe el archivo de eventos: {}".format(event_file)

print("Project root:", PROJECT_ROOT)
print("Match ID:", MATCH_ID)
print("Event file found:", event_file.name)
print("Data directory exists:", DATA_DIR.exists())
print("Notebook directory exists:", NOTEBOOK_DIR.exists())

Project root: c:\Users\kokec\OneDrive\Escritorio\SportsAnalytics\Projects\Statsbomb_open_data
Match ID: 69299
Event file found: 69299.json
Data directory exists: True
Notebook directory exists: True


### Find and retrieve match metadata

In [16]:
def find_match_metadata(match_id, matches_dir):
    """
    Find a match record in the StatsBomb match metadata files.
    """
    for file_path in matches_dir.rglob("*.json"):
        with file_path.open("r", encoding="utf-8") as file:
            matches = json.load(file)

        for match in matches:
            if match.get("match_id") == match_id:
                return match

    raise ValueError(
        f"Match ID {match_id} was not found in {matches_dir}"
    )

match_data = find_match_metadata(MATCH_ID, MATCHES_DIR)
match_data.keys()

dict_keys(['match_id', 'match_date', 'kick_off', 'competition', 'season', 'home_team', 'away_team', 'home_score', 'away_score', 'match_status', 'match_status_360', 'last_updated', 'last_updated_360', 'metadata', 'match_week', 'competition_stage', 'stadium', 'referee'])

In [22]:
print("Competition:", match_data.get("competition"))
print("Season:", match_data.get("season"))
print("Home team:", match_data.get("home_team"))
print("Away team:", match_data.get("away_team"))
print("Scores:", match_data.get("home_score"), match_data.get("away_score"))

Competition: {'competition_id': 11, 'country_name': 'Spain', 'competition_name': 'La Liga'}
Season: {'season_id': 22, 'season_name': '2010/2011'}
Home team: {'home_team_id': 217, 'home_team_name': 'Barcelona', 'home_team_gender': 'male', 'home_team_group': None, 'country': {'id': 214, 'name': 'Spain'}, 'managers': [{'id': 36, 'name': 'Josep Guardiola i Sala', 'nickname': 'Pep Guardiola', 'dob': '1971-01-18', 'country': {'id': 214, 'name': 'Spain'}}]}
Away team: {'away_team_id': 220, 'away_team_name': 'Real Madrid', 'away_team_gender': 'male', 'away_team_group': None, 'country': {'id': 214, 'name': 'Spain'}, 'managers': [{'id': 383, 'name': 'José Mario Felix dos Santos Mourinho', 'nickname': 'José Mourinho', 'dob': '1963-01-26', 'country': {'id': 183, 'name': 'Portugal'}}]}
Scores: 5 0


### Validate selected team

In [27]:
home_team = match_data["home_team"]["home_team_name"]
away_team = match_data["away_team"]["away_team_name"]

### Display match context

In [34]:
match_context = pd.DataFrame(
    {
        "Field": [
            "Competition",
            "Season",
            "Date",
            "Home team",
            "Away team",
            "Score",
            "Match ID",
            "Team analyzed",
        ],
        "Value": [
            match_data.get("competition", {}).get("competition_name"),
            match_data.get("season", {}).get("season_name"),
            match_data.get("match_date"),
            home_team,
            away_team,
            f"{match_data.get('home_score')} - {match_data.get('away_score')}",
            MATCH_ID,
            TEAM_NAME,
        ],
    }
)

display(match_context)

,Field,Value
0,Competition,La Liga
1,Season,2010/2011
2,Date,2010-11-29
3,Home team,Barcelona
4,Away team,Real Madrid
5,Score,5 - 0
6,Match ID,69299
7,Team analyzed,Barcelona


## 4. Analysis Dataset

The original StatsBomb event data contains information about all recorded actions in a football match.

To answer the research question, the raw events are transformed into a shot-level analytical dataset through the following steps:

### Transformation from raw events to analytical data

```text
1. StatsBombs event JSON
2. Load match events
2. Convert the event into a pandas Dataframe
3. Filter event type.name = "Shot"
4. Filter team.name == TEAM_NAME
5. Select relevant variables
6. Shot-level analytical dataset

### Variables selected

The analytical dataset retains variables directly related to the research question.

| Variable | Role in analysis | Reason for inclusion |
|---|---|---|
| `team.name` | Team identification | Identifies the team performing the shot |
| `player.name` | Player identification | Identifies the player taking the shot |
| `location` | Spatial analysis | Identifies the shot's starting coordinates |
| `shot.outcome.name` | Outcome analysis | Describes the result of the shot |
| `shot.statsbomb_xg` | Shot quality | Provides the StatsBomb expected goals value associated with the shot |

### Variables not used

Several variables available in the original event data are not incluided because they are not required to answer the current question.

For example, `shot.end_location` is not used in the main analysis because the research question concerns where the shot originated rather than where the ball ended.

Similarly, variables describing previous events or the construction of the attack are not included in this analysis because the objective is not to investigate how the shot was generated.

These variables may be relevant to future analysis, but they are outside the scope of the current research question.

**aquí va un código en python para importar la estructura exacta del archivo**

In [50]:
#sys.path.append(str(NOTEBOOK_DIR))
import shot_map

events = shot_map.load_events(event_file)

df = pd.json_normalize(events)

print(f"Total events loaded: {len(df):,}")
print(f"Available columns: {len(df.columns):,}")

shots = df.loc[df["type.name"].eq("Shot")].copy()

shots_team = shots.loc[
    shots["team.name"].eq(TEAM_NAME)
].copy()

print(f"Total shots in match: {len(shots):,}")
print(f"Shots by {TEAM_NAME}: {len(shots_team):,}")

selected_columns = [
    "team.name",
    "player.name",
    "minute",
    "second",
    "location",
    "shot.outcome.name",
    "shot.statsbomb_xg",
]

missing_columns = [
    column for column in selected_columns
    if column not in shots_team.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

shots_team = shots_team[selected_columns].copy()

assert shots_team["team.name"].eq(TEAM_NAME).all(), (
    "The analytical dataset contains shots from another team."
)

assert shots_team["location"].notna().all(), (
    "Some shots have missing starting coordinates."
)

assert shots_team["shot.outcome.name"].notna().all(), (
    "Some shots have missing outcomes."
)

print("Team filter validated.")
print("Shot locations checked.")
print("Shot outcomes checked.")

print(f"Match ID: {MATCH_ID}")
print(f"Team analyzed: {TEAM_NAME}")
print(f"Number of shots: {len(shots_team)}")
print(f"Number of variables: {len(shots_team.columns)}")

display(shots_team.head())

Total events loaded: 3,975
Available columns: 123
Total shots in match: 19
Shots by Barcelona: 14
Team filter validated.
Shot locations checked.
Shot outcomes checked.
Match ID: 69299
Team analyzed: Barcelona
Number of shots: 14
Number of variables: 7


,team.name,player.name,minute,second,location,shot.outcome.name,shot.statsbomb_xg
218,Barcelona,Lionel Andrés Messi Cuccittini,5,30,"[114.6, 58.7]",Post,0.007253
238,Barcelona,Pedro Eliezer Rodríguez Ledesma,5,49,"[96.8, 48.0]",Blocked,0.031406
247,Barcelona,Daniel Alves da Silva,6,18,"[86.7, 55.9]",Off T,0.005492
420,Barcelona,Xavier Hernández Creus,9,26,"[113.9, 41.2]",Goal,0.426909
813,Barcelona,Pedro Eliezer Rodríguez Ledesma,17,41,"[117.5, 41.3]",Goal,0.895359


In [51]:
#import inspect
#import shot_map

#print("Module location:", shot_map.__file__)
#print("Project root used by module:", shot_map.project_root)
#print("Current event file:", shot_map.file_path)
#print("load_events signature:", inspect.signature(shot_map.load_events))
#print(inspect.getsource(shot_map.load_events))

## 5. Methodology & Variable Selection

### Unit of analysis

The unit of analysis is an individual Barcelona shot event. Each observation in the analytical dataset therefore represents one shot performed by Barcelona during the match.

### Spatial representation

Shot locations are recorded using StatsBomb's pitch coordinate system, with a pitch dimension of 120 x 80.

For the spatial analysis, the pitch is divided into 18 rectangular zones using:

- 6 horizontal divisions along the x-axis.
- 3 vertical division along the y-axis.

This produces a 6 x 3 grid that allows shot locations to be summarized consistently by area.

### Shot outcome

Shot outcomes are retained from the original StatsBomb event data.

For visualization purposes, outcomes are grouped into four categories:

- Goal
- Saved
- Blocked
- Off Target / Other

The grouping is intented to simplify the visual encoding while preserving the distinction between goals, saved shots, blocked shots, and others non-goal outcomes 

### Expected goals

The analysis also uses `shot.statsbomb_xg`, when available, as a measure of the estimated probability that a shot results in a goal according to the StatsBomb model.

xG is used as a descriptive measure of shot quality and is not interpreted as a direct measure of finishing ability or player performance.